In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def crop_fundus_border(img_path, save_path = None):
    """
    More robust border cropping for fundus images where the retina
    is not perfectly circular. Uses threshold + convex hull.
    """

    # Read image (color)
    img = cv2.imread(img_path)
    original = img.copy()

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Smooth to remove small noise
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)

    # Threshold to isolate bright (non-black) areas
    _, thresh = cv2.threshold(blurred, 15, 255, cv2.THRESH_BINARY)

    # Find contours of non-black regions
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        print("No fundus area detected — returning original.")
        resized_original = cv2.resize(original, new_size)
        return resized_original

    # Combine all contours with a convex hull
    all_points = np.vstack(contours)
    hull = cv2.convexHull(all_points)

    # Get bounding rectangle of convex hull
    x, y, w, h = cv2.boundingRect(hull)

    # Crop
    cropped = original[y:y+h, x:x+w]
    return cropped

def resize(image):
    new_size = (299, 299)
    image = cv2.resize(cropped,new_size)
    return image


    # Optional save
    if save_path:
        img_name = os.path.basename(img_path)
        saved_img_path = os.path.join(save_path, img_name)
        cv2.imwrite(saved_img_path, cropped)

crop_img = FunctionTransformer(crop_fundus_border)
resize_img = FunctionTransformer(resize)

img_transformer = Pipeline([("Croper", crop_img),("resizer", resize_img)])